## Analyzing Whether MSE is Meaningful Without Normalizing Embeddings

In the non-contrastive setting, MSE barely budges and is already on the order of 1e-2. I wonder if that's just because the embedding norms are so small that the model doesn't really care about MSE.

In [16]:
import dotenv
import functools
import os
import pathlib

import torch
import torch.nn.functional as F
import datasets as hf_datasets
from mole_jepa import model_io, nfs_registry
from mole_jepa.data import transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

_ = dotenv.load_dotenv()

In [22]:
ds = hf_datasets.load_dataset(
    "clip-benchmark/wds_flickr30k",
    split="test",
    streaming=True,
)
ds = ds.take(512)



def collate(tokenize, image_transform, examples):
    pixels = torch.stack([image_transform(ex["jpg"]) for ex in examples])
    ids, masks = zip(*[tokenize(ex["txt"]) for ex in examples])
    return pixels, torch.stack(ids), torch.stack(masks)

In [24]:
for model_name in [
    "vit_small_miniml_jepa_frozen_v2",
    "vit_small_miniml_jepa_unfrozen_v2",
    "vit_small_miniml_infonce_frozen_v2",
]:
    print(model_name)
    entry = nfs_registry.get_entry(model_name)
    model = model_io.load_model(model_name, map_location=DEVICE).to(DEVICE)
    
    tokenize = transforms.build_tokenizer(
        entry.config.text_encoder_model_name, max_length=64
    )
    image_transform = transforms.build_image_transform(
        entry.config.image_encoder_model_name, train=False
    )

    cfn = functools.partial(collate, tokenize, image_transform)
    loader = torch.utils.data.DataLoader(
        list(ds), batch_size=128, collate_fn=cfn, num_workers=0
    )

    all_z_v, all_z_t, all_z_hat_t = [], [], []

    with torch.inference_mode():
        for pixels, input_ids, attn_mask in loader:
            out = model(pixels.to(DEVICE), input_ids.to(DEVICE), attn_mask.to(DEVICE))
            all_z_v.append(out.z_v.cpu())
            all_z_t.append(out.z_t.cpu())
            all_z_hat_t.append(out.z_hat_t.cpu())
    
    z_v = torch.cat(all_z_v)
    z_t = torch.cat(all_z_t)
    z_hat_t = torch.cat(all_z_hat_t)

    for name, z in [("z_v", z_v), ("z_t", z_t), ("z_hat_t", z_hat_t)]:
        norms = z.norm(dim=-1)
        print(f"{name:>10}  mean={norms.mean():.4f}  std={norms.std():.4f}  "
              f"min={norms.min():.4f}  max={norms.max():.4f}")
    
    print()
    mse_raw  = F.mse_loss(z_hat_t, z_t).item()
    mse_norm = F.mse_loss(F.normalize(z_hat_t, dim=-1), F.normalize(z_t, dim=-1)).item()
    cos_sim  = F.cosine_similarity(z_t, z_hat_t, dim=-1).mean().item()
    print(f"MSE (raw):        {mse_raw:.6f}")
    print(f"MSE (normalized): {mse_norm:.6f}")
    print(f"Cosine sim (mean):{cos_sim:.4f}")

    z_t_norm = F.normalize(z_t, dim=-1)
    gram = z_t_norm @ z_t_norm.T
    N = len(z_t_norm)
    off_diag_mask = ~torch.eye(N, dtype=torch.bool)
    print(f"z_t inter-sample cosine sim: mean={gram[off_diag_mask].mean():.4f}  std={gram[off_diag_mask].std():.4f}")
    

vit_small_miniml_jepa_frozen_v2


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 101153.74it/s]
[transformers] ViTModel LOAD REPORT from: WinKawaks/vit-small-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 42743.97it/s]


       z_v  mean=46.0637  std=2.1997  min=39.8668  max=54.0575
       z_t  mean=22.3285  std=3.4821  min=7.3341  max=32.6356
   z_hat_t  mean=19.8133  std=3.7332  min=7.0108  max=32.8861

MSE (raw):        0.060397
MSE (normalized): 0.000103
Cosine sim (mean):0.9606
z_t inter-sample cosine sim: mean=0.8387  std=0.1678
vit_small_miniml_jepa_unfrozen_v2


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 25374.20it/s]
[transformers] ViTModel LOAD REPORT from: WinKawaks/vit-small-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 27313.23it/s]


       z_v  mean=31.4381  std=2.4924  min=24.9028  max=39.3883
       z_t  mean=23.0396  std=3.3158  min=6.3594  max=32.2287
   z_hat_t  mean=21.1181  std=3.0498  min=9.3384  max=30.2790

MSE (raw):        0.040508
MSE (normalized): 0.000064
Cosine sim (mean):0.9756
z_t inter-sample cosine sim: mean=0.8387  std=0.1778
vit_small_miniml_infonce_frozen_v2


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 82749.32it/s]
[transformers] ViTModel LOAD REPORT from: WinKawaks/vit-small-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 26421.22it/s]


       z_v  mean=49.2422  std=4.2732  min=36.0634  max=60.9495
       z_t  mean=4.2403  std=0.5323  min=3.0654  max=6.2692
   z_hat_t  mean=15.9733  std=0.3874  min=14.5257  max=17.1448

MSE (raw):        0.356200
MSE (normalized): 0.002603
Cosine sim (mean):0.0003
z_t inter-sample cosine sim: mean=0.2013  std=0.1655


Well, looks like my hypothesis was wrong. The normalized MSE being small relative to the unnormalized MSE indicates that the model isn't artificially gaming the loss with small-norm embeddings. Rather, the more interesting (and unfortunate) observation is that the non-contrastive models are literally aligning everything. We can see this in the mean cosine similarity and $z_t$ inter-sample cosine similarity. There's only a gap of about 0.12, which is certainly not enough to aid retrieval.

So, we have collapse, which is exactly what SIGReg is supposed to combat. Looking at the 04 loss curves, the text SIGReg term is quite high and nothing is penalizing it hard enough. My solution here is to separate out the $\lambda$ regularization term for image and text embeddings, and significantly increase the $\lambda_{\text{text}}$ hyperparameter from 0.05 to something like 0.5. By pushing harder on regularization, one should be able to avoid representation collapse. Or so I hope. We'll see.